<style>
table {
  margin-left: 0 !important;
  margin-right: auto !important;
}
th, td {
  text-align: left !important;
}
</style>


## 02-1 · Part 3: Case C and Decision Domains

**The snack order must distinguish fruit measured by weight from boxes ordered as whole counts.**

Part 2 showed how requirements determine feasibility. This part formulates the school-event snack order from 01-2 and changes only which decision types are included.

### 1 · Carry forward the snack-order case

Let \(z\) be the number of 8-portion snack boxes and let \(p\) be the kilograms of fruit, which provide 4 portions per kilogram. Boxes cost \$12 each, and fruit costs \$7 per kilogram. The event needs at least 30 portions, with at most 5 boxes and 8 kg of fruit available.

The decision domain must record that \(z\) is a whole count while \(p\) is a measured amount.

### 2 · Formulate the combined order

Use the decision vector

> $\displaystyle x=\begin{bmatrix}z\\p\end{bmatrix}\in\{0,1,\ldots,5\}\times[0,8].$

The direct response is \(y=[C(x),P(x)]^{\mathsf T}\), where cost is \(C(x)=7p+12z\) and portions are \(P(x)=4p+8z\). The formulation is

> $\displaystyle \underset{x}{\operatorname{minimize}}\quad f(y)=C(x)=7p+12z$
>
> $\displaystyle \text{subject to}\quad g_1(x)=30-P(x)\le0,$
>
> $\displaystyle x\in\{0,1,\ldots,5\}\times[0,8].$

The portion constraint determines whether an order is sufficient. The domain determines whether each numerical value is an allowed action.

### 3 · Compare continuous, discrete, and mixed variants

| Variant | Decision domain | Type |
|:---|:---|:---|
| Fruit only | \(p\in[0,8]\) | Continuous |
| Boxes only | \(z\in\{0,1,\ldots,5\}\) | Discrete |
| Fruit and boxes | \((z,p)\in\{0,1,\ldots,5\}\times[0,8]\) | Mixed |

A continuous variable may take any real value in its interval. A discrete variable may take only listed or countable values. A mixed decision vector contains both.

Each variant minimizes the cost of the included item types and requires at least 30 portions. Only the allowed decision values change.

The next calculation enumerates every allowed box count and samples fruit in 0.25 kg increments. It checks the decision domain and portion requirement before comparing cost.

In [ ]:
import numpy as np

MAX_FRUIT = 8.0
MAX_BOXES = 5
REQUIRED_PORTIONS = 30.0


def evaluate_order(box_count, fruit_kg):
    decision = (float(box_count), float(fruit_kg))
    cost = 12.0 * decision[0] + 7.0 * decision[1]
    portions = 8.0 * decision[0] + 4.0 * decision[1]
    feasible = (
        0 <= decision[0] <= MAX_BOXES
        and decision[0].is_integer()
        and 0.0 <= decision[1] <= MAX_FRUIT
        and portions >= REQUIRED_PORTIONS
    )
    return {"x": decision, "cost": cost, "portions": portions, "feasible": feasible}

In [ ]:
fruit_grid = np.arange(0.0, MAX_FRUIT + 0.125, 0.25)
records = [
    evaluate_order(box_count, fruit_kg)
    for box_count in range(MAX_BOXES + 1)
    for fruit_kg in fruit_grid
]
best_grid = min(
    (record for record in records if record["feasible"]),
    key=lambda record: record["cost"],
)

print(
    f"Best sampled order: z={best_grid['x'][0]:.0f} boxes, "
    f"p={best_grid['x'][1]:.2f} kg"
)
print(f"Portions={best_grid['portions']:.0f}, cost={best_grid['cost']:.2f} dollars")

The result is the best candidate on the stated fruit grid. The box domain itself is genuinely discrete, but the 0.25 kg fruit grid is only a search choice. Fruit remains a continuous decision in the formulation.

### 4 · Keep domain and search method separate

| Statement | What it describes |
|:---|:---|
| \(p\in[0,8]\) | The real fruit decision is continuous |
| Evaluate \(p=0,0.25,\ldots,8\) | The demonstration samples a finite search grid |
| \(z\in\{0,1,\ldots,5\}\) | The real box decision is discrete |

The combined case is a **generally constrained, mixed, single-objective, linear, direct algebraic, deterministic optimization problem**. Because its objective and constraints are linear and it contains an integer variable, it is a mixed-integer linear program (MILP).

**Diagnosis.** Why does using a finer grid for \(p\) improve the search resolution without changing the problem from mixed to discrete?

### Takeaway

Classify decision values from the real action, not from the search code:

> **measured amount → continuous · whole count or named option → discrete · both in one decision vector → mixed**

A finite grid gives the best sampled candidate, not a guaranteed continuous optimum. Part 4 keeps continuous decisions and changes the number of objectives.